# 11.2 检索 (Retrieval)

检索是RAG的核心环节，从知识库中找到与查询最相关的文档。

本节涵盖：稠密检索、稀疏检索、混合检索、重排序

## 1. 稠密检索与稀疏检索

**稠密检索**：用双编码器将查询和文档映射到同一向量空间，通过向量相似度检索。
- DPR、E5、BGE等模型
- 优势：语义理解能力强

**稀疏检索**：基于词项匹配的检索方法。
- BM25：经典稀疏检索，基于TF-IDF改进
- SPLADE：学习稀疏表示，结合语义和词项匹配

**混合检索**：结合稠密和稀疏检索的优势，通常效果最好。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class DualEncoder(nn.Module):
    def __init__(self, d=64, d_out=16):
        super().__init__()
        self.q_enc = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, d_out))
        self.d_enc = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, d_out))
    def encode_query(self, q):
        return F.normalize(self.q_enc(q), dim=-1)
    def encode_doc(self, d):
        return F.normalize(self.d_enc(d), dim=-1)

model = DualEncoder()
n_docs = 100
queries = torch.randn(3, 64)
docs = torch.randn(n_docs, 64)

q_emb = model.encode_query(queries)
d_emb = model.encode_doc(docs)
dense_scores = q_emb @ d_emb.T

def bm25_sim(q, d, k1=1.5, b=0.75):
    scores = torch.zeros(q.shape[0], d.shape[0])
    for i in range(q.shape[0]):
        for j in range(d.shape[0]):
            dot = (q[i] * d[j]).sum()
            norm_d = d[j].norm()
            tf = dot / (norm_d + 1e-8)
            scores[i, j] = tf * (k1 + 1) / (tf + k1)
    return scores

sparse_scores = bm25_sim(queries, docs)
alpha = 0.7
hybrid_scores = alpha * dense_scores + (1 - alpha) * sparse_scores

print('=== Retrieval Methods ===')
print(f'Queries: {queries.shape[0]}, Docs: {n_docs}')
print(f'\nDense top-3 for query 0: {dense_scores[0].topk(3).indices.tolist()}')
print(f'Sparse top-3 for query 0: {sparse_scores[0].topk(3).indices.tolist()}')
print(f'Hybrid top-3 for query 0: {hybrid_scores[0].topk(3).indices.tolist()}')
print(f'\nKey: Hybrid retrieval combines semantic (dense) + lexical (sparse) matching.')

## 2. 重排序 (Reranking)

**目的**：对初步检索结果进行精细排序

**基本原理**：先用快速检索器召回候选集，再用交叉编码器（Cross-Encoder）对每个查询-文档对进行精确打分重排序。

**Cross-Encoder vs Bi-Encoder**：
- Bi-Encoder：查询和文档分别编码，速度快但精度低
- Cross-Encoder：查询和文档一起编码，精度高但速度慢
- 实践：Bi-Encoder召回 + Cross-Encoder重排

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class CrossEncoder(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d * 2, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, query, doc):
        combined = torch.cat([query, doc], dim=-1)
        return self.net(combined).squeeze(-1)

cross_enc = CrossEncoder()
query = torch.randn(1, 64)
candidate_docs = torch.randn(10, 64)

bi_scores = F.cosine_similarity(query, candidate_docs, dim=-1)
bi_top5 = bi_scores.topk(5).indices

cross_scores = cross_enc(query.expand(5, -1), candidate_docs[bi_top5])
reranked = bi_top5[cross_scores.argsort(descending=True)]

print('=== Reranking ===')
print(f'Bi-Encoder top-5: {bi_top5.tolist()}')
print(f'Cross-Encoder scores: {cross_scores.detach().tolist()}')
print(f'Reranked order: {reranked.tolist()}')
print(f'\nKey: Cross-Encoder is slower but more accurate for final ranking.')
print(f'Typical pipeline: Bi-Encoder (recall 100) -> Cross-Encoder (rerank top-10).')